In [ ]:
!pip install mlflow optuna optuna-integration -q

In [ ]:
import pandas as pd
import numpy as np
import tensorflow as tf
import mlflow
import mlflow.tensorflow
import optuna
from tqdm import tqdm

MLFLOW_EXPERIMENT = "two_tower_recommender"
BATCH_SIZE        = 4096
AUTOTUNE          = tf.data.AUTOTUNE
N_TRIALS          = 20
MAX_EPOCHS        = 40
EVAL_USERS        = 500

mlflow.set_experiment(MLFLOW_EXPERIMENT)
print("TF:", tf.__version__, "| MLflow:", mlflow.__version__, "| Optuna:", optuna.__version__)

In [ ]:
train_df = pd.read_parquet("train_df_128.parquet")
val_df   = pd.read_parquet("val_df_128.parquet")

overview_cols = [c for c in train_df.columns if c.startswith('overview_')]
affinity_cols = [c for c in train_df.columns if c.startswith('affinity_')]
genre_cols    = [c for c in train_df.columns if c.startswith('genre_')]

all_df = pd.concat([train_df, val_df], ignore_index=True)
user_id_map  = {uid: i for i, uid in enumerate(sorted(all_df['userId'].unique()))}
movie_id_map = {mid: i for i, mid in enumerate(sorted(all_df['movieId'].unique()))}

for df in [train_df, val_df]:
    df['user_idx']  = df['userId'].map(user_id_map)
    df['movie_idx'] = df['movieId'].map(movie_id_map)

n_users  = len(user_id_map)
n_movies = len(movie_id_map)

USER_FEATS = ['user_mean_rating','user_rating_std','user_rating_count','rating_deviation'] + affinity_cols
ITEM_FEATS = ['cf_mean_rating','cf_rating_std','log_cf_count','log_popularity',
              'log_vote_count','release_year','runtime','vote_average',
              'in_collection','collection_size'] + genre_cols + overview_cols

for df in [train_df, val_df]:
    df[USER_FEATS + ITEM_FEATS] = df[USER_FEATS + ITEM_FEATS].fillna(0)

user_feat_dim = len(USER_FEATS)
item_feat_dim = len(ITEM_FEATS)
N_OVERVIEW    = 128
N_OTHER       = item_feat_dim - N_OVERVIEW

print(f"Users:{n_users:,} Movies:{n_movies:,} | UF:{user_feat_dim} IF:{item_feat_dim}")

In [ ]:
def make_dataset(df, shuffle=False):
    ds = tf.data.Dataset.from_tensor_slices({
        'user_id':    df['user_idx'].values.astype(np.int32),
        'user_feats': df[USER_FEATS].values.astype(np.float32),
        'item_id':    df['movie_idx'].values.astype(np.int32),
        'item_feats': df[ITEM_FEATS].values.astype(np.float32),
    })
    if shuffle:
        ds = ds.shuffle(len(df), seed=None)
    return ds.batch(BATCH_SIZE).prefetch(AUTOTUNE)

train_ds = make_dataset(train_df, shuffle=True)
val_ds   = make_dataset(val_df)

In [ ]:
def build_user_tower(emb_dim=128, hidden=128, feat_proj=64, dropout=0.4, l2=1e-4):
    uid_in  = tf.keras.Input(shape=(), name='user_id', dtype='int32')
    feat_in = tf.keras.Input(shape=(user_feat_dim,), name='user_feats')

    u = tf.keras.layers.Embedding(n_users, emb_dim, embeddings_initializer='he_normal')(uid_in)
    u = tf.keras.layers.Flatten()(u)
    f = tf.keras.layers.Dense(feat_proj, activation='relu')(feat_in)
    f = tf.keras.layers.LayerNormalization()(f)

    x = tf.keras.layers.Concatenate()([u, f])
    for units in [hidden, hidden // 2]:
        x = tf.keras.layers.Dense(units, kernel_regularizer=tf.keras.regularizers.l2(l2))(x)
        x = tf.keras.layers.LayerNormalization()(x)
        x = tf.keras.layers.ReLU()(x)
        x = tf.keras.layers.Dropout(dropout)(x)
    x = tf.keras.layers.Dense(emb_dim)(x)
    x = tf.keras.layers.LayerNormalization()(x)
    x = tf.keras.layers.Lambda(lambda v: tf.math.l2_normalize(v, axis=1))(x)
    return tf.keras.Model([uid_in, feat_in], x, name='user_tower')


def build_item_tower(emb_dim=128, hidden=128, overview_proj=32, dropout=0.4, l2=1e-4):
    iid_in  = tf.keras.Input(shape=(), name='item_id', dtype='int32')
    feat_in = tf.keras.Input(shape=(item_feat_dim,), name='item_feats')

    i = tf.keras.layers.Embedding(n_movies, emb_dim, embeddings_initializer='he_normal')(iid_in)
    i = tf.keras.layers.Flatten()(i)

    other    = tf.keras.layers.Lambda(lambda x: x[:, :N_OTHER])(feat_in)
    overview = tf.keras.layers.Lambda(lambda x: x[:, N_OTHER:])(feat_in)
    ov_proj  = tf.keras.layers.Dense(overview_proj, activation='relu')(overview)

    x = tf.keras.layers.Concatenate()([i, other, ov_proj])
    for units in [hidden, hidden // 2]:
        x = tf.keras.layers.Dense(units, kernel_regularizer=tf.keras.regularizers.l2(l2))(x)
        x = tf.keras.layers.LayerNormalization()(x)
        x = tf.keras.layers.ReLU()(x)
        x = tf.keras.layers.Dropout(dropout)(x)
    x = tf.keras.layers.Dense(emb_dim)(x)
    x = tf.keras.layers.LayerNormalization()(x)
    x = tf.keras.layers.Lambda(lambda v: tf.math.l2_normalize(v, axis=1))(x)
    return tf.keras.Model([iid_in, feat_in], x, name='item_tower')

In [ ]:
class TwoTowerModel(tf.keras.Model):
    def __init__(self, user_model, item_model, init_temp=0.07):
        super().__init__()
        self.user_model = user_model
        self.item_model = item_model
        self.log_temp   = tf.Variable(np.log(init_temp), dtype=tf.float32, trainable=True)

    @property
    def temperature(self):
        return tf.exp(self.log_temp)

    def call(self, inputs, training=False):
        u = self.user_model([inputs['user_id'], inputs['user_feats']], training=training)
        i = self.item_model([inputs['item_id'], inputs['item_feats']], training=training)
        return u, i

    def compute_loss(self, u, i):
        u = tf.math.l2_normalize(u, axis=1)
        i = tf.math.l2_normalize(i, axis=1)
        logits = tf.matmul(u, i, transpose_b=True) / self.temperature
        bs     = tf.shape(logits)[0]
        labels = tf.range(bs)
        mask   = tf.cast(logits > 0.8, tf.float32) * (1.0 - tf.eye(bs))
        logits = logits - mask * 1e9
        loss_fn = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True)
        return (loss_fn(labels, logits) + loss_fn(labels, tf.transpose(logits))) / 2.0

    def recall_at_k(self, u, i, k=10):
        logits = tf.matmul(tf.math.l2_normalize(u, axis=1),
                           tf.math.l2_normalize(i, axis=1), transpose_b=True)
        bs    = tf.shape(u)[0]
        top_k = tf.math.top_k(logits, k=tf.minimum(k, bs)).indices
        hits  = tf.reduce_any(top_k == tf.expand_dims(tf.range(bs), 1), axis=1)
        return tf.reduce_mean(tf.cast(hits, tf.float32))

    def train_step(self, data):
        with tf.GradientTape() as tape:
            u, i = self(data, training=True)
            loss = self.compute_loss(u, i)
        self.optimizer.apply_gradients(zip(tape.gradient(loss, self.trainable_variables),
                                           self.trainable_variables))
        return {'loss': loss, 'recall@10': self.recall_at_k(u, i)}

    def test_step(self, data):
        u, i = self(data, training=False)
        return {'loss': self.compute_loss(u, i), 'recall@10': self.recall_at_k(u, i)}

In [ ]:
def evaluate_model(user_model, item_model, k=10, n_users=EVAL_USERS, seed=42):
    np.random.seed(seed)

    movie_lookup   = (train_df[['movie_idx','movieId'] + ITEM_FEATS]
                      .drop_duplicates('movie_idx').sort_values('movie_idx').reset_index(drop=True))
    all_idxs       = movie_lookup['movie_idx'].values.astype(np.int32)
    all_feats      = movie_lookup[ITEM_FEATS].values.astype(np.float32)
    all_item_embs  = item_model.predict([all_idxs, all_feats], batch_size=512, verbose=0)

    idx_to_row     = {idx: r for r, idx in enumerate(all_idxs)}
    row_to_idx     = {r: idx for r, idx in enumerate(all_idxs)}

    user_feat_map  = train_df.sort_values('timestamp').groupby('user_idx')[USER_FEATS].last()
    mean_uf        = user_feat_map.mean().values.astype(np.float32)

    item_pop       = np.log1p(train_df.groupby('movie_idx').size().reindex(all_idxs).fillna(0).values.astype(np.float32))
    pop_bias       = (item_pop - item_pop.mean()) / (item_pop.std() + 1e-8)

    user_pos       = val_df.groupby('user_idx')['movie_idx'].apply(set).to_dict()
    user_seen      = train_df.groupby('user_idx')['movie_idx'].apply(set).to_dict()

    sampled = np.random.choice(list(user_pos.keys()), min(n_users, len(user_pos)), replace=False)
    recall, hits, ndcg = [], [], []

    for uid in tqdm(sampled):
        uf    = user_feat_map.loc[uid].values.astype(np.float32) if uid in user_feat_map.index else mean_uf
        u_emb = user_model.predict([np.array([uid], np.int32), uf.reshape(1,-1)], verbose=0).reshape(-1)

        scores = all_item_embs @ u_emb
        scores = (scores - scores.mean()) / (scores.std() + 1e-8) + 0.05 * pop_bias
        seen_rows = [idx_to_row[i] for i in user_seen.get(uid, set()) if i in idx_to_row]
        scores[seen_rows] = -np.inf

        k_eff   = min(k, len(scores))
        top_k   = np.argpartition(-scores, k_eff)[:k_eff]
        top_k   = top_k[np.argsort(-scores[top_k])]
        preds   = [row_to_idx[r] for r in top_k]
        truth   = user_pos[uid]
        if not truth:
            continue

        h = len(set(preds) & truth)
        recall.append(h / len(truth))
        hits.append(int(h > 0))
        dcg  = sum(1/np.log2(i+2) for i,p in enumerate(preds) if p in truth)
        idcg = sum(1/np.log2(i+2) for i in range(min(len(truth), k)))
        ndcg.append(dcg/idcg if idcg > 0 else 0)

    return {f'recall@{k}': np.mean(recall), f'hit@{k}': np.mean(hits), f'ndcg@{k}': np.mean(ndcg)}

In [ ]:
def objective(trial):
    params = {
        'emb_dim':      trial.suggest_categorical('emb_dim', [64, 128, 256]),
        'hidden':       trial.suggest_categorical('hidden', [128, 256, 512]),
        'feat_proj':    trial.suggest_categorical('feat_proj', [32, 64, 128]),
        'overview_proj':trial.suggest_categorical('overview_proj', [16, 32, 64]),
        'dropout':      trial.suggest_float('dropout', 0.1, 0.5, step=0.1),
        'l2':           trial.suggest_float('l2', 1e-5, 1e-3, log=True),
        'lr':           trial.suggest_float('lr', 1e-4, 1e-2, log=True),
        'init_temp':    trial.suggest_float('init_temp', 0.05, 0.2),
    }

    with mlflow.start_run(run_name=f"trial_{trial.number}", nested=True):
        mlflow.log_params(params)

        user_m = build_user_tower(
            emb_dim=params['emb_dim'], hidden=params['hidden'],
            feat_proj=params['feat_proj'], dropout=params['dropout'], l2=params['l2'])
        item_m = build_item_tower(
            emb_dim=params['emb_dim'], hidden=params['hidden'],
            overview_proj=params['overview_proj'], dropout=params['dropout'], l2=params['l2'])

        model = TwoTowerModel(user_m, item_m, init_temp=params['init_temp'])
        model.compile(optimizer=tf.keras.optimizers.Adam(params['lr']))

        callbacks = [
            tf.keras.callbacks.EarlyStopping(patience=7, restore_best_weights=True, monitor='val_loss'),
            tf.keras.callbacks.ReduceLROnPlateau(patience=3, factor=0.5, monitor='val_loss', min_delta=0.001),
        ]

        history = model.fit(train_ds, validation_data=val_ds,
                            epochs=MAX_EPOCHS, callbacks=callbacks, verbose=0)

        best_epoch   = int(np.argmin(history.history['val_loss']))
        best_val_loss = float(history.history['val_loss'][best_epoch])
        best_recall  = float(history.history.get('val_recall@10', [0])[best_epoch])

        mlflow.log_metrics({'val_loss': best_val_loss, 'val_recall@10': best_recall,
                            'best_epoch': best_epoch})

        # Full evaluation @ 10
        eval_metrics = evaluate_model(user_m, item_m, k=10)
        mlflow.log_metrics(eval_metrics)
        print(f"Trial {trial.number} | val_loss={best_val_loss:.4f} | {eval_metrics}")

    return best_val_loss

In [ ]:
with mlflow.start_run(run_name="optuna_study"):
    study = optuna.create_study(direction='minimize',
                                 sampler=optuna.samplers.TPESampler(seed=42))
    study.optimize(objective, n_trials=N_TRIALS, show_progress_bar=True)

    best = study.best_trial
    mlflow.log_params({f"best_{k}": v for k, v in best.params.items()})
    mlflow.log_metric("best_val_loss", best.value)

    print("\n=== Best Trial ===")
    print(f"  val_loss : {best.value:.4f}")
    print(f"  params   : {best.params}")

In [ ]:
bp = study.best_params

user_model = build_user_tower(
    emb_dim=bp['emb_dim'], hidden=bp['hidden'],
    feat_proj=bp['feat_proj'], dropout=bp['dropout'], l2=bp['l2'])
item_model = build_item_tower(
    emb_dim=bp['emb_dim'], hidden=bp['hidden'],
    overview_proj=bp['overview_proj'], dropout=bp['dropout'], l2=bp['l2'])

two_tower = TwoTowerModel(user_model, item_model, init_temp=bp['init_temp'])
two_tower.compile(optimizer=tf.keras.optimizers.Adam(bp['lr']))

with mlflow.start_run(run_name="best_model_final"):
    mlflow.log_params(bp)

    history = two_tower.fit(
        train_ds, validation_data=val_ds, epochs=MAX_EPOCHS,
        callbacks=[
            tf.keras.callbacks.EarlyStopping(patience=7, restore_best_weights=True, monitor='val_loss'),
            tf.keras.callbacks.ReduceLROnPlateau(patience=3, factor=0.5, monitor='val_loss', min_delta=0.001),
        ]
    )

    for k in [1, 5, 10, 20, 50]:
        m = evaluate_model(user_model, item_model, k=k)
        mlflow.log_metrics(m)
        print(f"k={k:2d} | {m}")

    # Save towers
    user_model.save("user_model_best")
    item_model.save("item_model_best")
    mlflow.tensorflow.log_model(user_model, "user_model")
    mlflow.tensorflow.log_model(item_model, "item_model")
    print("\nModels saved & logged to MLflow.")

In [ ]:
# Load saved models (skip if already in memory from Cell 10)
user_model = tf.keras.models.load_model("user_model_best")
item_model = tf.keras.models.load_model("item_model_best")

movie_lookup  = (train_df[['movie_idx','movieId'] + ITEM_FEATS]
                 .drop_duplicates('movie_idx')
                 .sort_values('movie_idx')
                 .reset_index(drop=True))

all_idxs      = movie_lookup['movie_idx'].values.astype(np.int32)
all_feats     = movie_lookup[ITEM_FEATS].values.astype(np.float32)
all_item_embs = item_model.predict([all_idxs, all_feats], batch_size=512, verbose=1)

row_to_movieid = dict(enumerate(movie_lookup['movieId'].values))
user_feat_map  = train_df.sort_values('timestamp').groupby('user_idx')[USER_FEATS].last()
mean_uf        = user_feat_map.mean().values.astype(np.float32)
user_seen      = train_df.groupby('user_idx')['movie_idx'].apply(set).to_dict()

item_pop  = np.log1p(train_df.groupby('movie_idx').size().reindex(all_idxs).fillna(0).values.astype(np.float32))
pop_bias  = (item_pop - item_pop.mean()) / (item_pop.std() + 1e-8)

print(f"Item index ready: {all_item_embs.shape}")

In [ ]:
def recommend(user_id, k=10, pop_weight=0.05, exclude_seen=True):
    """
    user_id      : original userId (not idx)
    k            : number of recommendations
    pop_weight   : popularity re-ranking weight
    exclude_seen : filter out already-watched movies
    """
    # Map to internal index
    if user_id not in user_id_map:
        raise ValueError(f"userId {user_id} not found. Use a known user.")
    uid = user_id_map[user_id]

    # Get user features
    uf = user_feat_map.loc[uid].values.astype(np.float32) if uid in user_feat_map.index else mean_uf

    # Embed user
    u_emb = user_model.predict(
        [np.array([uid], np.int32), uf.reshape(1, -1)], verbose=0
    ).reshape(-1)

    # Score all items
    scores = all_item_embs @ u_emb
    scores = (scores - scores.mean()) / (scores.std() + 1e-8)
    scores += pop_weight * pop_bias

    # Mask seen
    if exclude_seen:
        seen_rows = [i for i, idx in enumerate(all_idxs) if idx in user_seen.get(uid, set())]
        scores[seen_rows] = -np.inf

    # Top-k
    k_eff  = min(k, (scores > -np.inf).sum())
    top_k  = np.argpartition(-scores, k_eff)[:k_eff]
    top_k  = top_k[np.argsort(-scores[top_k])]

    results = pd.DataFrame({
        'rank':    range(1, len(top_k) + 1),
        'movieId': [row_to_movieid[r] for r in top_k],
        'score':   scores[top_k].round(4),
    })
    return results

In [ ]:
# Pick any userId from your data
sample_user = train_df['userId'].iloc[0]

recs = recommend(sample_user, k=10)
print(f"Top-10 recommendations for user {sample_user}:\n")
print(recs.to_string(index=False))